In [ ]:
!pip install "numpy<2"
!pip install xarray
!pip install matplotlib
!pip install cartopy
!pip install netcdf4 h5netcdf

In [ ]:
# Plot setup: configure fonts and default save folder directly in the notebook
import xarray as xr
import re
import matplotlib.pyplot as plt
from matplotlib import rcParams
from pathlib import Path

PLOT_SAVE_DIR = Path('plots')
PLOT_SAVE_DIR.mkdir(parents=True, exist_ok=True)


def _safe_filename(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r"[^0-9A-Za-z._-]+", "_", name)
    return name or "figure"


def configure_plot_styles(fontsize: int = 16):
    rcParams['font.size'] = fontsize
    rcParams['axes.titlesize'] = fontsize + 2
    rcParams['axes.labelsize'] = fontsize
    rcParams['xtick.labelsize'] = max(10, fontsize - 2)
    rcParams['ytick.labelsize'] = max(10, fontsize - 2)
    rcParams['legend.fontsize'] = max(10, fontsize - 2)
    rcParams['figure.titlesize'] = fontsize + 4


def save_all_figures(prefix: str = None, formats=('png', 'pdf'), dpi: int = 150, save_dir: Path = None):
    target_dir = Path(save_dir or PLOT_SAVE_DIR)
    target_dir.mkdir(parents=True, exist_ok=True)

    for num in plt.get_fignums():
        fig = plt.figure(num)
        title = ''
        if getattr(fig, '_suptitle', None) is not None:
            title = fig._suptitle.get_text() or ''
        if not title:
            axes = fig.get_axes()
            if axes:
                title = axes[0].get_title() or ''
        if not title:
            title = f'figure_{num}'

        filename = _safe_filename(title)
        if prefix:
            filename = f'{prefix}_{filename}'

        for fmt in formats:
            path = target_dir / f'{filename}.{fmt}'
            if fmt.lower() == 'pdf':
                fig.savefig(path, format='pdf')
            else:
                fig.savefig(path, dpi=dpi, format=fmt)


configure_plot_styles(fontsize=16)
print(f'Set up plot fonts and save folder: {PLOT_SAVE_DIR}')

In [ ]:
DATA_DIRS = {
    '187.5_1x': {
        'data': '/gpfs/data/fs72044/icon15/simulations/slab187_1co2/slab187_1co2_atm_2d_2000_2025_remap_sic.nc',
        'sic': '/gpfs/data/fs72044/icon15/simulations/slab187_1co2/slab187_1co2_atm_2d_2000_2025_remap_sic.nc',
        'color': 'blue',
        'linestyle': '-'
    },
    '187.5_2x': {
        'data': '/gpfs/data/fs72044/icon15/simulations/slab187_2co2/slab187_2co2_atm_2d_2000_2035_remap_sic.nc',
        'sic': '/gpfs/data/fs72044/icon15/simulations/slab187_2co2/slab187_2co2_atm_2d_2000_2035_remap_sic.nc',
        'color': 'blue',
        'linestyle': '--'
    },
    '375_1x': {
        'data': '/gpfs/data/fs72044/icon18/experiments/s2026/slabcn2sea375/slab375_1co2_2d_2000_2035.cell_area.r360x180.nc',
        'sic': '/gpfs/data/fs72044/icon18/experiments/s2026/slabcn2sea375/slab375_1co2_2d_seaice.nc',
        'color': 'orange',
        'linestyle': '-'
    },
    '375_2x': {
        'data': '/gpfs/data/fs72044/icon15/simulations/slab375_2co2/slab375_2co2_atm_2d_2000_2035_remap.nc',
        'sic': '/gpfs/data/fs72044/icon15/simulations/slab375_2co2/slab375_2co2_atm_2d_2000_2035_remap_sic.nc',
        'color': 'orange',
        'linestyle': '--'
    },
    '560_1x': {
        'data': '/gpfs/data/fs72044/icon28/Simulations/slabcn560/slabcn560_1co2_2d_2000-2035_remapped.nc',
        'sic': '/gpfs/data/fs72044/icon28/Simulations/slabcn560/slab560_1_sic.nc',
        'color': 'green',
        'linestyle': '-'
    },
    '560_2x': {
        'data': '/gpfs/data/fs72044/icon28/Simulations/slabcn560doubledco2/slabcn560doubledco2_2d_2000_2019_remapped.nc',
        'sic': '/gpfs/data/fs72044/icon28/Simulations/slabcn560doubledco2/slab560_2_sic.nc',
        'color': 'green',
        'linestyle': '--'
    },
}

datasets = {
    name: {
        'ds': xr.open_dataset(cfg['data']),
        'color': cfg['color'],
        'linestyle': cfg['linestyle']
    }
    for name, cfg in DATA_DIRS.items()
}

sic_datasets = {
    name: xr.open_dataset(cfg['sic'])
    for name, cfg in DATA_DIRS.items()
}

ctrl_path = '/gpfs/data/fs72044/avoigt_teach/experiments/s2026/slabctr/slabctr_atm_2d_ml_1979-2035.remapcon-r360x180.nc'
ds_ctrl = xr.open_dataset(ctrl_path)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
from scipy.stats import ttest_ind
import pandas as pd
from cartopy.util import add_cyclic_point

# ── Time and dataset helpers ─────────────────────────────────────────────────

def yyyymmdd_to_datetimeindex(time_values):
    """Convert YYYYMMDD integer-like values to pandas DatetimeIndex."""
    ints = time_values.astype(int)
    text = [f"{v:08d}" for v in ints]
    return pd.to_datetime(text, format="%Y%m%d")


def complete_years_only(da):
    """Keep only years with 12 monthly entries."""
    years = da.time.dt.year
    unique_years, counts = np.unique(years.values, return_counts=True)
    full_years = unique_years[counts == 12]
    return da.sel(time=da.time.dt.year.isin(full_years))


def fix_lon(da):
    """Add a cyclic point on the longitude axis so global maps close cleanly."""
    data, lons = add_cyclic_point(da.values, coord=da.lon.values)
    coords = {k: v for k, v in da.coords.items() if k != 'lon'}
    coords['lon'] = lons
    return da.__class__(data, dims=da.dims, coords=coords, attrs=da.attrs)


def prepare_da(ds, var, fix_lon_grid=False):
    da = ds[var]
    da = da.assign_coords(time=yyyymmdd_to_datetimeindex(da.time.values)).sortby('time')
    da = complete_years_only(da)
    return fix_lon(da) if fix_lon_grid else da


def annual_mean(ds, var):
    return prepare_da(ds, var).resample(time='YE').mean()


def last_n_year_mean(ds, var, n=5, fix_lon_grid=True):
    ann = prepare_da(ds, var, fix_lon_grid=fix_lon_grid).resample(time='YE').mean()
    ann = ann.isel(time=slice(-n, None))
    years = ann.time.dt.year.values
    label = f"{years[0]}–{years[-1]}" if len(years) > 1 else str(years[0])
    return ann.mean(dim='time'), label


def mean_global_annual(ds, var):
    return annual_mean(ds, var).mean(dim=['lon', 'lat'])


def compute_pval(ds_exp, ds_ctr, var, n=5):
    def _last_n(ds):
        da = prepare_da(ds, var, fix_lon_grid=True)
        ann = da.resample(time='YE').mean()
        return ann.isel(time=slice(-n, None))

    exp_ann = _last_n(ds_exp)
    ctr_ann = _last_n(ds_ctr)
    if not (exp_ann.lon.equals(ctr_ann.lon) and exp_ann.lat.equals(ctr_ann.lat)):
        ctr_ann = ctr_ann.interp_like(exp_ann)

    _, pval = ttest_ind(exp_ann.values, ctr_ann.values, axis=0,
                        equal_var=False, nan_policy='omit')
    return xr.DataArray(pval, coords={'lat': exp_ann.lat, 'lon': exp_ann.lon}, dims=['lat', 'lon'])


def overlay_insignificance(ax, pval_da):
    insig = (pval_da >= 0.05).values
    lon2d, lat2d = np.meshgrid(pval_da.lon.values, pval_da.lat.values)
    ax.scatter(lon2d[insig], lat2d[insig], s=0.4, color='gray', alpha=0.5,
               transform=ccrs.PlateCarree(), zorder=5)


def add_latlon_ticks(ax):
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                      linewidth=0.4, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlocator = plt.FixedLocator([-120, -60, 0, 60, 120])
    gl.ylocator = plt.FixedLocator([-60, -30, 0, 30, 60])
    gl.xlabel_style = {'size': 7}
    gl.ylabel_style = {'size': 7}


def plot_global_annual_series(ax, data, label, color, linestyle):
    years = data.time.dt.year.values.astype(int)
    ax.plot(years, data.values, label=label, color=color, linestyle=linestyle, linewidth=2)


def make_map_ax(fig, rows, cols, idx):
    ax = fig.add_subplot(rows, cols, idx, projection=ccrs.EqualEarth())
    ax.coastlines()
    add_latlon_ticks(ax)
    return ax


def draw_contour(ax, da, cmap, levels=30, vmin=None, vmax=None, cb_label=''):
    im = ax.contourf(da.lon, da.lat, da, transform=ccrs.PlateCarree(),
                     cmap=cmap, levels=levels, vmin=vmin, vmax=vmax)
    cb = plt.colorbar(im, ax=ax, orientation='vertical', pad=0.05)
    if cb_label:
        cb.set_label(cb_label)
    return im


def align_to_grid(da, target):
    if not (da.lon.equals(target.lon) and da.lat.equals(target.lat)):
        return da.interp_like(target)
    return da

# ── Radiation variables and dataset metadata ─────────────────────────────────

radiation_vars = [
    'rsdt', 'rsut', 'rsutcs', 'rlut', 'rlutcs',
    'rsds', 'rsdscs', 'rlds', 'rldscs',
    'rsus', 'rsuscs', 'rlus'
]

var_names = {
    'rsdt':   'TOA Incident Shortwave Radiation',
    'rsut':   'TOA Outgoing Shortwave Radiation',
    'rsutcs': 'TOA Outgoing Clear-Sky Shortwave Radiation',
    'rlut':   'TOA Outgoing Longwave Radiation',
    'rlutcs': 'TOA Outgoing Clear-Sky Longwave Radiation',
    'rsds':   'Surface Downwelling Shortwave Radiation',
    'rsdscs': 'Surface Downwelling Clear-Sky Shortwave Radiation',
    'rlds':   'Surface Downwelling Longwave Radiation',
    'rldscs': 'Surface Downwelling Clear-Sky Longwave Radiation',
    'rsus':   'Surface Upwelling Shortwave Radiation',
    'rsuscs': 'Surface Upwelling Clear-Sky Shortwave Radiation',
    'rlus':   'Surface Upwelling Longwave Radiation'
}


In [ ]:
def plot_radiation_time_series(radiation_vars, datasets, control_ds):
    for var in radiation_vars:
        fig, ax = plt.subplots(figsize=(12, 6), dpi=150)

        if var in control_ds.data_vars:
            data = mean_global_annual(control_ds, var)
            plot_global_annual_series(ax, data, 'control', 'black', '-')

        for name, props in datasets.items():
            ds = props['ds']
            if ds is None or var not in ds.data_vars:
                continue

            data = mean_global_annual(ds, var)
            plot_global_annual_series(ax, data, name, props['color'], props['linestyle'])

        ax.set_xlabel('Year')
        ax.set_ylabel(f'{var} (W/m²)')
        ax.set_title(f'Annual Mean Time Series: {var_names[var]}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xlim(2000, 2035)
        plt.tight_layout()
        plt.show()


def plot_sic_time_series(sic_datasets, datasets, control_ds):
    fig, ax = plt.subplots(figsize=(12, 6), dpi=150)

    if 'sic' in control_ds.data_vars:
        data = mean_global_annual(control_ds, 'sic')
        plot_global_annual_series(ax, data, 'control', 'black', '-')

    for name, ds in sic_datasets.items():
        if ds is None or 'sic' not in ds.data_vars:
            continue

        data = mean_global_annual(ds, 'sic')
        plot_props = datasets.get(name, {'color': 'gray', 'linestyle': '-'})
        plot_global_annual_series(ax, data, name, plot_props['color'], plot_props['linestyle'])

    ax.set_xlabel('Year')
    ax.set_ylabel('SIC (fraction)')
    ax.set_title('Annual Mean Time Series: Sea Ice Content (SIC)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(2000, 2035)
    plt.tight_layout()
    plt.show()

plot_radiation_time_series(radiation_vars, datasets, ds_ctrl)
plot_sic_time_series(sic_datasets, datasets, ds_ctrl)


In [ ]:
def plot_radiation_anomaly_maps(radiation_vars, datasets, control_ds):
    for var in radiation_vars:
        valid_experiments = [name for name, props in datasets.items()
                             if props['ds'] is not None and var in props['ds'].data_vars]
        if not valid_experiments:
            continue

        total_plots = 1 + len(valid_experiments)
        n_cols = 2
        n_rows = (total_plots + n_cols - 1) // n_cols

        fig = plt.figure(figsize=(24, 16), dpi=150)

        if var in control_ds.data_vars:
            ctrl_data, ctrl_yrs = last_n_year_mean(control_ds, var)
            ax = make_map_ax(fig, n_rows, n_cols, 1)
            draw_contour(ax, ctrl_data, cmap='afmhot', levels=30, cb_label='W/m²')
            ax.set_title(f'{var_names[var]}\nControl  [{ctrl_yrs}]')

        plot_idx = 2
        for name in valid_experiments:
            ds = datasets[name]['ds']
            exp_data, exp_yrs = last_n_year_mean(ds, var)
            ctrl_mean, ctrl_yrs = last_n_year_mean(control_ds, var)
            ctrl_mean = align_to_grid(ctrl_mean, exp_data)

            anomaly = exp_data - ctrl_mean
            vmax = float(np.nanmax(np.abs(anomaly.values)))

            ax = make_map_ax(fig, n_rows, n_cols, plot_idx)
            draw_contour(ax, anomaly, cmap='PuOr_r', levels=30, vmin=-vmax, vmax=vmax, cb_label='W/m²')
            pval = compute_pval(ds, control_ds, var)
            overlay_insignificance(ax, pval)
            ax.set_title(f'{var_names[var]}\n{name}  [{exp_yrs}]  –  Control  [{ctrl_yrs}]  (gray = p≥0.05)')
            plot_idx += 1

        plt.tight_layout()
        plt.show()

plot_radiation_anomaly_maps(radiation_vars, datasets, ds_ctrl)


In [ ]:
def plot_side_by_side_maps(radiation_vars, datasets, control_ds, n=5):
    for var in radiation_vars:
        for name, props in datasets.items():
            ds = props['ds']
            if ds is None or var not in ds.data_vars or var not in control_ds.data_vars:
                continue

            exp_data, exp_yrs = last_n_year_mean(ds, var)
            ctrl_data, ctrl_yrs = last_n_year_mean(control_ds, var)
            ctrl_data = align_to_grid(ctrl_data, exp_data)
            anomaly = exp_data - ctrl_data
            vmax = float(np.nanmax(np.abs(anomaly.values)))

            fig = plt.figure(figsize=(24, 7), dpi=150)

            ax1 = make_map_ax(fig, 1, 3, 1)
            draw_contour(ax1, ctrl_data, cmap='afmhot', levels=30, cb_label='W/m²')
            ax1.set_title(f'{var_names[var]}\nControl  [{ctrl_yrs}]')

            ax2 = make_map_ax(fig, 1, 3, 2)
            draw_contour(ax2, exp_data, cmap='afmhot', levels=30, cb_label='W/m²')
            ax2.set_title(f'{var_names[var]}\n{name}  [{exp_yrs}]')

            ax3 = make_map_ax(fig, 1, 3, 3)
            draw_contour(ax3, anomaly, cmap='PuOr_r', levels=30, vmin=-vmax, vmax=vmax, cb_label='W/m²')
            pval = compute_pval(ds, control_ds, var)
            overlay_insignificance(ax3, pval)
            ax3.set_title(f'{var_names[var]}\n{name}  [{exp_yrs}]  –  Control  [{ctrl_yrs}]  (gray = p≥0.05)')

            plt.tight_layout()
            plt.show()

plot_side_by_side_maps(radiation_vars, datasets, ds_ctrl)


In [ ]:
if 'sic' not in ds_ctrl.data_vars:
    print("WARNING: ds_ctrl does not contain 'sic'; skipping SIC vs control maps.")
else:
    ctrl_sic_data, ctrl_sic_yrs = last_n_year_mean(ds_ctrl, 'sic')

    for name, ds_exp_sic in sic_datasets.items():
        if ds_exp_sic is None or 'sic' not in ds_exp_sic.data_vars:
            print(f"Skipping {name}: 'sic' variable not found")
            continue

        exp_sic_data, exp_sic_yrs = last_n_year_mean(ds_exp_sic, 'sic')
        ctrl_sic_interp = align_to_grid(ctrl_sic_data, exp_sic_data)
        anomaly = exp_sic_data - ctrl_sic_interp
        vmax = float(np.nanmax(np.abs(anomaly.values)))

        pval = compute_pval(ds_exp_sic, ds_ctrl, 'sic')

        fig = plt.figure(figsize=(24, 7), dpi=150)

        ax1 = make_map_ax(fig, 1, 3, 1)
        draw_contour(ax1, ctrl_sic_data, cmap='Blues_r', levels=30, vmin=0, vmax=1, cb_label='fraction (0–1)')
        ax1.set_title(f'Sea Ice Content\nControl  [{ctrl_sic_yrs}]')

        ax2 = make_map_ax(fig, 1, 3, 2)
        draw_contour(ax2, exp_sic_data, cmap='Blues_r', levels=30, vmin=0, vmax=1, cb_label='fraction (0–1)')
        ax2.set_title(f'Sea Ice Content\n{name}  [{exp_sic_yrs}]')

        ax3 = make_map_ax(fig, 1, 3, 3)
        draw_contour(ax3, anomaly, cmap='PuOr_r', levels=30, vmin=-vmax, vmax=vmax, cb_label='fraction (0–1)')
        overlay_insignificance(ax3, pval)
        ax3.set_title(f'SIC Anomaly\n{name}  [{exp_sic_yrs}]  –  Control  [{ctrl_sic_yrs}]  (gray = p≥0.05)')

        plt.tight_layout()
        plt.show()


In [ ]:
import matplotlib.animation as animation
from IPython.display import Image, display

if 'sic' not in ds_ctrl.data_vars:
    print("WARNING: ds_ctrl does not contain 'sic'; skipping SIC animations.")
else:
    ann_ctrl_full = prepare_da(ds_ctrl, 'sic', fix_lon_grid=True).resample(time='YE').mean()

    for name, ds_exp_sic in sic_datasets.items():
        if ds_exp_sic is None or 'sic' not in ds_exp_sic.data_vars:
            print(f"Skipping {name}: 'sic' not found")
            continue

        ann_exp = prepare_da(ds_exp_sic, 'sic', fix_lon_grid=True).resample(time='YE').mean()

        yrs_exp = set(ann_exp.time.dt.year.values)
        yrs_ctrl = set(ann_ctrl_full.time.dt.year.values)
        common_years = sorted(yrs_exp & yrs_ctrl)

        if not common_years:
            print(f"No overlapping years for {name} vs control, skipping animation")
            continue

        ann_exp_sel = ann_exp.sel(time=ann_exp.time.dt.year.isin(common_years))
        ann_ctrl_sel = ann_ctrl_full.sel(time=ann_ctrl_full.time.dt.year.isin(common_years))
        ann_ctrl_sel = align_to_grid(ann_ctrl_sel, ann_exp_sel)

        anomalies = ann_exp_sel - ann_ctrl_sel
        vmax_anom = float(np.nanmax(np.abs(anomalies.values)))

        fig = plt.figure(figsize=(16, 5))
        ax1 = fig.add_subplot(1, 3, 1, projection=ccrs.EqualEarth())
        ax2 = fig.add_subplot(1, 3, 2, projection=ccrs.EqualEarth())
        ax3 = fig.add_subplot(1, 3, 3, projection=ccrs.EqualEarth())

        for ax in [ax1, ax2, ax3]:
            ax.coastlines()
            add_latlon_ticks(ax)

        frame_ctrl = ann_ctrl_sel.isel(time=0)
        frame_exp = ann_exp_sel.isel(time=0)
        frame_an = anomalies.isel(time=0)

        cf1 = ax1.contourf(frame_ctrl.lon, frame_ctrl.lat, frame_ctrl,
                           transform=ccrs.PlateCarree(), cmap='Blues_r', levels=20, vmin=0, vmax=1)
        cf2 = ax2.contourf(frame_exp.lon, frame_exp.lat, frame_exp,
                           transform=ccrs.PlateCarree(), cmap='Blues_r', levels=20, vmin=0, vmax=1)
        cf3 = ax3.contourf(frame_an.lon, frame_an.lat, frame_an,
                           transform=ccrs.PlateCarree(), cmap='PuOr_r', levels=20, vmin=-vmax_anom, vmax=vmax_anom)

        cb1 = plt.colorbar(cf1, ax=ax1, orientation='vertical', pad=0.05); cb1.set_label('fraction (0–1)')
        cb2 = plt.colorbar(cf2, ax=ax2, orientation='vertical', pad=0.05); cb2.set_label('fraction (0–1)')
        cb3 = plt.colorbar(cf3, ax=ax3, orientation='vertical', pad=0.05); cb3.set_label('fraction (0–1)')

        title1 = ax1.set_title(f'SIC  Control  [{common_years[0]}]')
        title2 = ax2.set_title(f'SIC  {name}  [{common_years[0]}]')
        title3 = ax3.set_title(f'SIC Anomaly  {name} – Control  [{common_years[0]}]')
        plt.tight_layout()

        def update(frame_idx):
            yr = common_years[frame_idx]
            fc = ann_ctrl_sel.isel(time=frame_idx)
            fe = ann_exp_sel.isel(time=frame_idx)
            fa = anomalies.isel(time=frame_idx)

            for ax in [ax1, ax2, ax3]:
                for coll in ax.collections:
                    coll.remove()

            ax1.contourf(fc.lon, fc.lat, fc, transform=ccrs.PlateCarree(),
                         cmap='Blues_r', levels=20, vmin=0, vmax=1)
            ax2.contourf(fe.lon, fe.lat, fe, transform=ccrs.PlateCarree(),
                         cmap='Blues_r', levels=20, vmin=0, vmax=1)
            ax3.contourf(fa.lon, fa.lat, fa, transform=ccrs.PlateCarree(),
                         cmap='PuOr_r', levels=20, vmin=-vmax_anom, vmax=vmax_anom)

            title1.set_text(f'SIC  Control  [{yr}]')
            title2.set_text(f'SIC  {name}  [{yr}]')
            title3.set_text(f'SIC Anomaly  {name} – Control  [{yr}]')
            return []

        ani = animation.FuncAnimation(fig, update, frames=len(common_years), interval=400, blit=False)
        gif_path = f'sic_animation_{name}_vs_ctrl.gif'
        ani.save(gif_path, writer='pillow', fps=2.5, dpi=100)
        plt.close(fig)

        print(f"Saved: {gif_path}  ({len(common_years)} frames, years {common_years[0]}–{common_years[-1]})")
        display(Image(filename=gif_path))
